# Solar Panel Detection Model Training & Testing
Make sure your runtime is set to GPU (Runtime > Change runtime type > T4 GPU).

## 1. Unzip the Dataset & Install Dependencies
Run this cell to extract the dataset folder and install ultralytics.

In [ ]:
!unzip -q -o dataset.zip
!pip install ultralytics
import ultralytics
ultralytics.checks()

## 2. Train the Model
Train the YOLOv8-OBB model using the dataset. You can change `epochs=50` to a higher number if you want to train it longer.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 nano OBB model
model = YOLO('yolov8n-obb.pt')

# Train the model
results = model.train(
    data='./dataset/data.yaml', 
    epochs=50, 
    imgsz=640,
    batch=8,
    patience=10,
    verbose=True
)

## 3. Test the Model (Upload & Predict)
Run this cell to upload any test image from your computer. The trained model will predict solar panels and display the result!

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from google.colab import files
from ultralytics import YOLO

print("Please upload an image to test:")
uploaded = files.upload()

if uploaded:
    # Get the file name
    file_name = list(uploaded.keys())[0]
    
    # Load your best trained model
    best_model = YOLO('runs/obb/train/weights/best.pt')
    
    # Run prediction
    results = best_model.predict(file_name, conf=0.25)
    
    # Display the result
    res_plotted = results[0].plot() 
    cv2_imshow(res_plotted)

## 4. Download the Best Weights
Run this cell to download `best.pt` to your local computer so you can use it in your web app.

In [ ]:
from google.colab import files
import os

# Check if best.pt exists and download it
weights_path = 'runs/obb/train/weights/best.pt'
if os.path.exists(weights_path):
    files.download(weights_path)
else:
    print('Could not find best.pt. Please check the runs/ directory on the left side panel.')

## 5. Deploy Live to Hugging Face Spaces
Run these cells to automatically create a live, shareable Gradio web app on Hugging Face using your newly trained model!

**Prerequisites:**
1. Create a free account at [Hugging Face](https://huggingface.co/)
2. Create a new Space (Settings -> Spaces -> Create New Space) and choose **Gradio** as the SDK.
3. Go to [Settings -> Access Tokens](https://huggingface.co/settings/tokens) and create a token with **Write** permissions.

In [ ]:
!pip install huggingface_hub gradio
from huggingface_hub import notebook_login

# This will prompt you to paste your Hugging Face write token
notebook_login()

Generate the deployment code for your web app:

In [ ]:
import os

os.makedirs('hf_space', exist_ok=True)

# Write the Gradio interface code (app.py)
app_code = """\
import gradio as gr
from ultralytics import YOLO
import cv2
import spaces

model = YOLO('best.pt')

@spaces.GPU
def predict_solar_panels(image):
    results = model.predict(image, conf=0.25)
    res_plotted = results[0].plot()
    return res_plotted

iface = gr.Interface(
    fn=predict_solar_panels,
    inputs=gr.Image(type="numpy", label="Upload Solar Panel Image"),
    outputs=gr.Image(type="numpy", label="Detection Result"),
    title="Solar Panel Detection YOLOv8",
    description="Upload an image to detect solar panels using a model trained on Google Colab!"
)
iface.launch()
"""
with open('hf_space/app.py', 'w') as f:
    f.write(app_code)

# Write requirements for Hugging Face
req_code = """\
ultralytics
gradio
opencv-python-headless
"""
with open('hf_space/requirements.txt', 'w') as f:
    f.write(req_code)

print("Deployment files created!")

Upload everything to your Space to make it live online! 

In [ ]:
from huggingface_hub import HfApi

# ==================================================================
# SPACE ID CONFIGURED FOR: smsaad001/solarpanel
# ==================================================================
SPACE_ID = 'smsaad001/solarpanel' 

api = HfApi()
print(f"Uploading files to Hugging Face Space: {SPACE_ID}...")

# Upload app.py
api.upload_file(
    path_or_fileobj="hf_space/app.py",
    path_in_repo="app.py",
    repo_id=SPACE_ID,
    repo_type="space"
)

# Upload requirements.txt
api.upload_file(
    path_or_fileobj="hf_space/requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=SPACE_ID,
    repo_type="space"
)

# Upload the trained weights
api.upload_file(
    path_or_fileobj="runs/obb/train/weights/best.pt",
    path_in_repo="best.pt",
    repo_id=SPACE_ID,
    repo_type="space"
)

print(f"\nSuccess! Your app is now building live at: https://huggingface.co/spaces/{SPACE_ID}")